In [ ]:
!pip install captum lime

In [ ]:
# 1. System & Utility Libraries
import os
import time
import csv
import psutil
import numpy as np
from PIL import Image

# 2. PyTorch Core & Vision Libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

# 3. Explainable AI (XAI) Libraries
from captum.attr import LayerGradCam, IntegratedGradients
from lime import lime_image

# 4. Google Colab Specifics
from google.colab import drive

# 5. Mount Drive (Executes immediately upon setup)
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
DRIVE_DATA_DIR = '/content/drive/MyDrive/CAPSTONE/Datasets/MEDMNIST'

DATASETS = [
    'pneumoniamnist_224.npz',
    'dermamnist_224.npz',
    'bloodmnist_224.npz',
    'chestmnist_224.npz',
    'pathmnist_224.npz'
]

In [ ]:
import zipfile
import os
import shutil
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms

class CustomMedMNISTDataset(Dataset):
    def __init__(self, npz_path, split='train', transform=None, extract_dir='/content/tmp_mmap'):
        self.transform = transform
        os.makedirs(extract_dir, exist_ok=True)

        base_name = os.path.basename(npz_path).replace('.npz', '')
        img_name = f'{split}_images.npy'
        lbl_name = f'{split}_labels.npy'

        self.img_path = os.path.join(extract_dir, f"{base_name}_{img_name}")
        self.lbl_path = os.path.join(extract_dir, f"{base_name}_{lbl_name}")

        # Stream the extraction to disk to bypass the 12GB RAM limit
        if not (os.path.exists(self.img_path) and os.path.exists(self.lbl_path)):
            print(f"Streaming {split} arrays to disk for {base_name}...")
            with zipfile.ZipFile(npz_path, 'r') as zip_ref:
                with zip_ref.open(img_name) as source, open(self.img_path, 'wb') as target:
                    shutil.copyfileobj(source, target)
                with zip_ref.open(lbl_name) as source, open(self.lbl_path, 'wb') as target:
                    shutil.copyfileobj(source, target)

        # Map the files to virtual memory to keep RAM usage near 0
        self.images = np.load(self.img_path, mmap_mode='r')
        self.labels = np.load(self.lbl_path, mmap_mode='r')

        self.is_grayscale = (len(self.images.shape) == 3)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_array = self.images[idx]
        labels = self.labels[idx].astype(np.float32)

        if self.is_grayscale:
            img = Image.fromarray(img_array, mode='L')
        else:
            img = Image.fromarray(img_array, mode='RGB')

        if self.transform:
            img = self.transform(img)

        return img, labels

def prepare_dataloaders(npz_file_name, data_dir, batch_size=64, num_bench_samples=100):
    npz_path = os.path.join(data_dir, npz_file_name)

    data_transform = transforms.Compose([
        transforms.Lambda(lambda x: x.convert('RGB') if x.mode != 'RGB' else x),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = CustomMedMNISTDataset(npz_path, split='train', transform=data_transform)
    val_dataset = CustomMedMNISTDataset(npz_path, split='val', transform=data_transform)
    test_dataset = CustomMedMNISTDataset(npz_path, split='test', transform=data_transform)

    train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    bench_subset = Subset(test_dataset, list(range(num_bench_samples)))
    bench_loader = DataLoader(dataset=bench_subset, batch_size=1, shuffle=False)

    return train_loader, val_loader, bench_loader

In [ ]:
def train_model(model, train_loader, val_loader, task_type, device, epochs=10):
    model = model.to(device)

    if task_type == 'multi-label, binary-class':
        criterion = nn.BCEWithLogitsLoss()
    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)


            if task_type != 'multi-label, binary-class':
                targets = targets.squeeze().long()
            else:
                targets = targets.float()

            optimizer.zero_grad()
            outputs = model(inputs)

            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch+1}/{epochs} - Training Loss: {epoch_loss:.4f}")

    return model

In [ ]:
class XAITelemetryEngine:
    def __init__(self, model, device):
        self.model = model.to(device)
        self.model.eval()
        self.device = device

        self.grad_cam = LayerGradCam(self.model, self.model.features[-1])
        self.ig = IntegratedGradients(self.model)
        self.lime_explainer = lime_image.LimeImageExplainer()

    def get_process_ram_mb(self):
        """Captures the exact RAM consumed by this specific Python process."""
        process = psutil.Process(os.getpid())
        return process.memory_info().rss / (1024 * 1024)

    def extract_gradcam(self, input_tensor, target_class):
        start_ram = self.get_process_ram_mb()
        start_time = time.perf_counter()

        attribution = self.grad_cam.attribute(input_tensor.to(self.device), target=target_class)

        elapsed_ms = (time.perf_counter() - start_time) * 1000
        peak_ram = self.get_process_ram_mb() - start_ram

        return attribution.detach().cpu(), elapsed_ms, peak_ram

    def extract_integrated_gradients(self, input_tensor, target_class, steps=50):
        start_ram = self.get_process_ram_mb()
        start_time = time.perf_counter()

        attribution = self.ig.attribute(input_tensor.to(self.device), target=target_class, n_steps=steps)

        elapsed_ms = (time.perf_counter() - start_time) * 1000
        peak_ram = self.get_process_ram_mb() - start_ram

        return attribution.detach().cpu(), elapsed_ms, peak_ram

    def extract_lime(self, input_tensor, target_class, num_samples=500):
        start_ram = self.get_process_ram_mb()
        start_time = time.perf_counter()

        img_np = input_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()

        def predict_fn(images):
            batch = torch.tensor(images).permute(0, 3, 1, 2).float().to(self.device)
            with torch.no_grad():
                logits = self.model(batch)
            return logits.cpu().numpy()

        explanation = self.lime_explainer.explain_instance(
            img_np, predict_fn, labels=(target_class,), num_samples=num_samples, top_labels=1
        )

        elapsed_ms = (time.perf_counter() - start_time) * 1000
        peak_ram = self.get_process_ram_mb() - start_ram

        return explanation, elapsed_ms, peak_ram

In [ ]:
def run_hardware_benchmark(xai_engine, bench_loader, csv_path="xai_benchmark_complete.csv"):
    # 1. Initialize CSV with the expanded headers (including AUC)
    with open(csv_path, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow([
            "Image_ID", "True_Label", "Algorithm",
            "Latency_ms", "RAM_Spike_MB", "Deletion_AUC", "Insertion_AUC"
        ])

    for idx, (input_tensor, targets) in enumerate(bench_loader):
        # Target Normalization
        if targets.ndim > 1 and targets.shape[1] > 1:
            target_class = torch.argmax(targets[0]).item()
        else:
            target_class = int(targets[0].item())

        print(f"\nProcessing Image {idx+1}/100 [True Label: {target_class}]")

        gc_attr, gc_lat, gc_ram = xai_engine.extract_gradcam(input_tensor, target_class)
        gc_del, gc_ins = evaluate_insertion_deletion(
            xai_engine.model, input_tensor, gc_attr, target_class, device=xai_engine.device
        )
        log_to_csv(csv_path, [idx, target_class, "Grad-CAM", gc_lat, gc_ram, gc_del, gc_ins])

        ig_attr, ig_lat, ig_ram = xai_engine.extract_integrated_gradients(input_tensor, target_class)
        ig_del, ig_ins = evaluate_insertion_deletion(
            xai_engine.model, input_tensor, ig_attr, target_class, device=xai_engine.device
        )
        log_to_csv(csv_path, [idx, target_class, "Integrated_Gradients", ig_lat, ig_ram, ig_del, ig_ins])


        lime_explanation, lime_lat, lime_ram = xai_engine.extract_lime(input_tensor, target_class)

        lime_mask = torch.tensor(
            lime_explanation.get_image_and_mask(
                lime_explanation.top_labels[0], positive_only=False, num_features=10, hide_rest=False
            )[1]
        ).float().unsqueeze(0).unsqueeze(0)

        lime_del, lime_ins = evaluate_insertion_deletion(
            xai_engine.model, input_tensor, lime_mask, target_class, device=xai_engine.device
        )
        log_to_csv(csv_path, [idx, target_class, "LIME", lime_lat, lime_ram, lime_del, lime_ins])

def log_to_csv(csv_path, row_data):
    with open(csv_path, mode='a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(row_data)

In [ ]:
def evaluate_insertion_deletion(model, input_tensor, attribution, target_class, steps=10, device='cpu'):
    # Flatten attribution and get sorted indices
    attr_flat = attribution.detach().cpu().numpy().flatten()

    # Map flat indices to 2D spatial dimensions (H, W)
    input_np = input_tensor.squeeze(0).cpu().numpy()
    channels, height, width = input_np.shape
    total_pixels = height * width

    # If attribution is channel-wise or has different dimensions, aggregate spatially
    if attribution.ndim == 4: # [1, C, H, W]
        spatial_attr = attribution.squeeze(0).mean(dim=0).detach().cpu().numpy()
    elif attribution.ndim == 3: # [C, H, W]
        spatial_attr = attribution.mean(dim=0).detach().cpu().numpy()
    else:
        spatial_attr = attribution.detach().cpu().numpy().squeeze()

    spatial_flat = spatial_attr.flatten()
    sorted_spatial_indices = np.argsort(spatial_flat)[::-1]

    step_size = max(1, total_pixels // steps)

    # 1. Deletion Metric (start from original image, gradually mask out highest-importance pixels)
    deletion_scores = []
    deletion_img = input_np.copy()
    for i in range(0, total_pixels, step_size):
        indices_to_mask = sorted_spatial_indices[:i]
        for idx in indices_to_mask:
            h_idx = idx // width
            w_idx = idx % width
            deletion_img[:, h_idx, w_idx] = 0.0 # apply zero baseline mask

        test_tensor = torch.tensor(deletion_img).unsqueeze(0).to(device)
        with torch.no_grad():
            output = model(test_tensor)
            # Support multi-class (softmax) or binary/multi-label (sigmoid)
            if output.shape[1] > 1:
                score = torch.softmax(output, dim=1)[0, target_class].item()
            else:
                score = torch.sigmoid(output)[0, 0].item()
        deletion_scores.append(score)

    # 2. Insertion Metric (start from empty baseline, gradually restore highest-importance pixels)
    insertion_scores = []
    insertion_img = np.zeros_like(input_np)
    for i in range(0, total_pixels, step_size):
        indices_to_restore = sorted_spatial_indices[:i]
        for idx in indices_to_restore:
            h_idx = idx // width
            w_idx = idx % width
            insertion_img[:, h_idx, w_idx] = input_np[:, h_idx, w_idx]

        test_tensor = torch.tensor(insertion_img).unsqueeze(0).to(device)
        with torch.no_grad():
            output = model(test_tensor)
            if output.shape[1] > 1:
                score = torch.softmax(output, dim=1)[0, target_class].item()
            else:
                score = torch.sigmoid(output)[0, 0].item()
        insertion_scores.append(score)

    # Calculate Area Under Curve (AUC)
    deletion_auc = float(np.mean(deletion_scores))
    insertion_auc = float(np.mean(insertion_scores))

    return deletion_auc, insertion_auc

In [ ]:

# 1. Define model builder function
def build_mobilenet_v3(num_classes):
    model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    return model

# 2. Correct the dataset profiles dictionary mapping
dataset_profiles = {
    'pneumoniamnist_224.npz': {'num_classes': 2, 'task_type': 'binary-class'},
    'dermamnist_224.npz': {'num_classes': 7, 'task_type': 'multi-class'},
    'bloodmnist_224.npz': {'num_classes': 8, 'task_type': 'multi-class'},
    'chestmnist_224.npz': {'num_classes': 14, 'task_type': 'multi-label, binary-class'},
    'pathmnist_224.npz': {'num_classes': 9, 'task_type': 'multi-class'}
}

EPOCHS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing pipeline on device: {DEVICE}")

# Ensure output directories exist
os.makedirs("/content/drive/MyDrive/CAPSTONE/Models/MobileNetV3/ModelWeights", exist_ok=True)
os.makedirs("/content/drive/MyDrive/CAPSTONE/Models/MobileNetV3/XAI_Benchmark", exist_ok=True)

# Loop over ALL datasets with memory optimization
for dataset_file in DATASETS:
    # Run for all datasets but focus on releasing memory iteratively
    profile = dataset_profiles[dataset_file]
    num_classes = profile['num_classes']
    task_type = profile['task_type']

    print(f"\n{'='*50}\nProcessing Dataset: {dataset_file}\n{'='*50}")

    try:
        # Clear memory before starting new dataset
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # Use 0 num_workers to reduce RAM overhead in Colab
        train_loader, val_loader, bench_loader = prepare_dataloaders(
            npz_file_name=dataset_file,
            data_dir=DRIVE_DATA_DIR,
            batch_size=32, # Slightly reduced batch size to lower peak VRAM
            num_bench_samples=100
        )
        # Force dataloader workers to 0 dynamically to prevent RAM leaks
        train_loader.num_workers = 0
        val_loader.num_workers = 0

        print("Building MobileNetV3-Small...")
        model = build_mobilenet_v3(num_classes=num_classes)

        print("Starting Training...")
        trained_model = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            task_type=task_type,
            device=DEVICE,
            epochs=EPOCHS
        )

        weight_path = f"/content/drive/MyDrive/CAPSTONE/Models/MobileNetV3/ModelWeights/mobilenetv3_{dataset_file.split('_')[0]}.pth"
        torch.save(trained_model.state_dict(), weight_path)
        print(f"Model saved to {weight_path}")

        print("Initializing XAI Telemetry Engine...")
        xai_engine = XAITelemetryEngine(model=trained_model, device=DEVICE)

        # Patch LIME explainer in the active engine instance to use fewer samples to prevent crash
        original_extract_lime = xai_engine.extract_lime
        def optimized_extract_lime(input_tensor, target_class, num_samples=100):
            return original_extract_lime(input_tensor, target_class, num_samples=num_samples)
        xai_engine.extract_lime = optimized_extract_lime

        csv_out_path = f"/content/drive/MyDrive/CAPSTONE/Models/MobileNetV3/XAI_Benchmark/xai_benchmark_{dataset_file.split('_')[0]}.csv"
        print(f"Running benchmarks. Logging to {csv_out_path}...")
        run_hardware_benchmark(xai_engine=xai_engine, bench_loader=bench_loader, csv_path=csv_out_path)

        print(f"Finished Benchmark for {dataset_file}!")

        # Cleanup variables to reclaim memory
        del train_loader, val_loader, bench_loader, model, trained_model, xai_engine
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as e:
        print(f"Error processing dataset {dataset_file}: {str(e)}")
        import traceback
        traceback.print_exc()
        continue

print("\nAll MedMNIST benchmarks completed successfully!")

Executing pipeline on device: cuda

Processing Dataset: pneumoniamnist_224.npz
Streaming train arrays to disk for pneumoniamnist_224...
Streaming val arrays to disk for pneumoniamnist_224...
Streaming test arrays to disk for pneumoniamnist_224...
Building MobileNetV3-Small...
Starting Training...


/tmp/ipykernel_43347/3675902500.py:44: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(img_array, mode='L')


Epoch 1/5 - Training Loss: 0.1151
Epoch 2/5 - Training Loss: 0.0464
Epoch 3/5 - Training Loss: 0.0339
Epoch 4/5 - Training Loss: 0.0239
Epoch 5/5 - Training Loss: 0.0200
Model saved to /content/drive/MyDrive/CAPSTONE/Models/MobileNetV3/ModelWeights/mobilenetv3_pneumoniamnist.pth
Initializing XAI Telemetry Engine...
Running benchmarks. Logging to /content/drive/MyDrive/CAPSTONE/Models/MobileNetV3/XAI_Benchmark/xai_benchmark_pneumoniamnist.csv...

Processing Image 1/100 [True Label: 1]


  0%|          | 0/100 [00:00<?, ?it/s]

/tmp/ipykernel_43347/3675902500.py:44: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(img_array, mode='L')


...

  0%|          | 0/100 [00:00<?, ?it/s]

/tmp/ipykernel_43347/3675902500.py:46: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(img_array, mode='RGB')



Processing Image 100/100 [True Label: 1]


  0%|          | 0/100 [00:00<?, ?it/s]

Finished Benchmark for pathmnist_224.npz!

All MedMNIST benchmarks completed successfully!


In [ ]:


# Ensure base configurations are defined
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 5

dataset_profiles = {
    'pneumoniamnist_224.npz': {'num_classes': 2, 'task_type': 'binary-class'},
    'dermamnist_224.npz': {'num_classes': 7, 'task_type': 'multi-class'},
    'bloodmnist_224.npz': {'num_classes': 8, 'task_type': 'multi-class'},
    'chestmnist_224.npz': {'num_classes': 14, 'task_type': 'multi-label, binary-class'},
    'pathmnist_224.npz': {'num_classes': 9, 'task_type': 'multi-class'}
}

# Target dataset for this dedicated run
target_dataset = 'pathmnist_224.npz'
profile = dataset_profiles[target_dataset]
num_classes = profile['num_classes']
task_type = profile['task_type']

print(f"\n{'='*50}\nProcessing Target Dataset: {target_dataset}\n{'='*50}")

try:
    # Clear memory
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Prepare loaders with 0 num_workers for Colab stability
    train_loader, val_loader, bench_loader = prepare_dataloaders(
        npz_file_name=target_dataset,
        data_dir=DRIVE_DATA_DIR,
        batch_size=32,
        num_bench_samples=100
    )
    train_loader.num_workers = 0
    val_loader.num_workers = 0

    print("Building MobileNetV3-Small...")
    model = build_mobilenet_v3(num_classes=num_classes)

    print("Starting Training...")
    trained_model = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        task_type=task_type,
        device=DEVICE,
        epochs=EPOCHS
    )

    weight_path = f"/content/drive/MyDrive/CAPSTONE/Models/MobileNetV3/ModelWeights/mobilenetv3_{target_dataset.split('_')[0]}.pth"
    torch.save(trained_model.state_dict(), weight_path)
    print(f"Model saved to {weight_path}")

    print("Initializing XAI Telemetry Engine...")
    xai_engine = XAITelemetryEngine(model=trained_model, device=DEVICE)

    # Apply optimized patch to LIME to keep benchmark footprint lightweight
    original_extract_lime = xai_engine.extract_lime
    def optimized_extract_lime(input_tensor, target_class, num_samples=100):
        return original_extract_lime(input_tensor, target_class, num_samples=num_samples)
    xai_engine.extract_lime = optimized_extract_lime

    csv_out_path = f"/content/drive/MyDrive/CAPSTONE/Models/MobileNetV3/XAI_Benchmark/xai_benchmark_{target_dataset.split('_')[0]}.csv"
    print(f"Running benchmarks. Logging to {csv_out_path}...")
    run_hardware_benchmark(xai_engine=xai_engine, bench_loader=bench_loader, csv_path=csv_out_path)

    print(f"Finished Benchmark for {target_dataset}!")

    # Free memory
    del train_loader, val_loader, bench_loader, model, trained_model, xai_engine
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:
    print(f"Error processing dataset {target_dataset}: {str(e)}")
    import traceback
    traceback.print_exc()
